# ComfyUI AI Influencer on Kaggle

This notebook sets up ComfyUI for AI influencer generation on Kaggle with T4 x2 GPUs.

### Instructions:
1. Enable T4 x2 GPU.
2. Turn on Internet Access.
3. Run cells sequentially.

In [ ]:
!git clone https://github.com/Dhruwkd/comfyui-kaggle-influencer.git
%cd comfyui-kaggle-influencer
!chmod +x scripts/*.sh
!./scripts/install_nodes.sh
!./scripts/download_models.sh

In [ ]:
# Install cloudflared binary to a local path (avoid "Text file busy" on /usr/local/bin)
!mkdir -p /kaggle/working/bin
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/bin/cloudflared
!chmod +x /kaggle/working/bin/cloudflared

import subprocess
import threading
import time
import os

# Verify ComfyUI exists
comfyui_path = "/kaggle/working/ComfyUI"
if not os.path.exists(comfyui_path):
    raise RuntimeError(f"ComfyUI not found at {comfyui_path}. Did cell 1 complete successfully?")

def run_tunnel():
    # Use the local binary, capture output to see the URL
    proc = subprocess.Popen(
        ["/kaggle/working/bin/cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    # Print tunnel URL as it appears
    for line in proc.stdout:
        print(f"[cloudflared] {line.strip()}")
        if "trycloudflare.com" in line:
            print(f"\n>>> TUNNEL URL: {line.strip()}\n")

tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
tunnel_thread.start()

# Give cloudflared time to start and establish tunnel
time.sleep(8)

# Start ComfyUI from the correct directory
os.chdir(comfyui_path)
print(f"Starting ComfyUI from {comfyui_path}...")
!python main.py --lowvram --port 8188